In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.nn import GINEConv, BatchNorm, Linear
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_recall_fscore_support,
)

DATA_DIR = Path('Data')
OUT_DIR = Path('outputs')
SEED = 1

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Rate features

In [ ]:
# using a big number to mark timestamps
BIG = 10_000_000  

def history_counts(ids, t, first_seen):
    keys = np.sort(ids * BIG + t)
    now = np.searchsorted(keys, ids * BIG + t)
    count_24h = now - np.searchsorted(keys, ids * BIG + np.maximum(t - 86400, 0))
    count_all = now - np.searchsorted(keys, ids * BIG)
    rate = count_all / np.maximum((t - first_seen[ids]) / 86400, 1.0)
    return count_24h, rate

Loading data and splitting

In [ ]:
csv_path = DATA_DIR / 'HI-Small_Trans.csv'
parquet_path = DATA_DIR / 'processed' / 'transactions.parquet'

if parquet_path.exists():
    print('Using preprocessed data from the parquet file')
    df = pd.read_parquet(parquet_path)
else:
    print('Preparing data from the raw CSV')
    df = pd.read_csv(csv_path, dtype=str)

    cols = list(df.columns)
    cols[2], cols[4] = 'From Account', 'To Account'
    df.columns = cols

    # seconds since the first transaction
    df['EdgeID'] = np.arange(len(df), dtype=np.int64)
    ts = pd.to_datetime(df['Timestamp'], format='%Y/%m/%d %H:%M')
    df['Timestamp'] = (ts - ts.min()).dt.total_seconds().astype(np.int64)
    df['hour'] = ts.dt.hour.astype(np.int64)
    df['day_of_week'] = ts.dt.dayofweek.astype(np.int64)

    # (Multi-GNN format_kaggle_files.py)
    currencies = pd.Index(pd.unique(
        np.column_stack([df['Receiving Currency'], df['Payment Currency']]).ravel()
    ))
    formats = pd.Index(pd.unique(df['Payment Format']))
    df['Received Currency'] = currencies.get_indexer(df['Receiving Currency'])
    df['Payment Format'] = formats.get_indexer(df['Payment Format'])

    from_key = df['From Bank'] + '_' + df['From Account']
    to_key = df['To Bank'] + '_' + df['To Account']
    nodes = pd.Index(pd.unique(np.column_stack([from_key, to_key]).ravel()))
    df['from_id'] = nodes.get_indexer(from_key)
    df['to_id'] = nodes.get_indexer(to_key)

    df['Amount Received'] = df['Amount Received'].astype(np.float64)
    df['Is Laundering'] = df['Is Laundering'].astype(np.int64)

    df = df[[
        'EdgeID',
        'from_id',
        'to_id',
        'Timestamp',
        'Amount Received',
        'Received Currency',
        'Payment Format',
        'Is Laundering',
        'hour',
        'day_of_week',
    ]]
    df = df.sort_values(['Timestamp', 'EdgeID'], kind='stable').reset_index(drop=True)

    # (Multi-GNN data_loading.py)
    t = df['Timestamp'].to_numpy()
    day = t // 86400
    daily = np.bincount(day)
    target_split = np.array([0.6, 0.2, 0.2])
    best_score = np.inf
    for i in range(len(daily)):
        for j in range(i + 1, len(daily)):
            props = np.array([daily[:i].sum(), daily[i:j].sum(), daily[j:].sum()]) / daily.sum()
            score = np.max(np.abs(props - target_split) / target_split)
            if score < best_score:
                best_score, train_end, val_end = score, i, j
    df['split'] = np.where(day < train_end, 'train', np.where(day < val_end, 'val', 'test'))
    print(f'Split days: train=[0, {train_end}), val=[{train_end}, {val_end}), test=[{val_end}, {len(daily)})')

    # rate features
    from_ids, to_ids = df['from_id'].to_numpy(), df['to_id'].to_numpy()
    first_seen = np.full(len(nodes), np.iinfo(np.int64).max)
    np.minimum.at(first_seen, np.concatenate([from_ids, to_ids]), np.concatenate([t, t]))
    sender_24h, sender_rate = history_counts(from_ids, t, first_seen)
    receiver_24h, receiver_rate = history_counts(to_ids, t, first_seen)
    df['sender_out_count_24h'] = sender_24h
    df['receiver_in_count_24h'] = receiver_24h
    df['sender_out_rate_lifetime'] = sender_rate
    df['receiver_in_rate_lifetime'] = receiver_rate

    parquet_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(parquet_path, index=False)

print(f"Transactions: {len(df):,} | Laundering: {df['Is Laundering'].sum():,}")
print(df['split'].value_counts())

Evaluation

In [ ]:
def report(name, y_val, val_prob, y_test, test_prob):
    precision, recall, thresholds = precision_recall_curve(y_val, val_prob)
    f1 = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros_like(precision),
        where=(precision + recall) > 0,
    )[:-1]
    threshold = thresholds[np.flatnonzero(f1 == f1.max())[-1]]  # highest threshold on ties
    y_pred = (test_prob >= threshold).astype(int)

    print(f'\n{name} Results:')
    print(f'Threshold chosen on validation: {threshold:.4f} (validation F1: {f1.max():.4f})')

    print('\n--- Confusion Matrix ---')
    cm = confusion_matrix(y_test, y_pred)
    print(f'True Negatives: {cm[0][0]}  |  False Positives: {cm[0][1]}')
    print(f'False Negatives: {cm[1][0]}    |  True Positives: {cm[1][1]}')

    print('\n--- Classification Report ---')
    print(classification_report(y_test, y_pred, digits=4, zero_division=0))

    auprc = average_precision_score(y_test, test_prob)
    print(f'Area Under the Precision-Recall Curve (AUPRC): {auprc:.4f}')

    p, r, f, _ = precision_recall_fscore_support(y_test, y_pred, average='binary', zero_division=0)
    return {'Model': name, 'Test AUPRC': auprc, 'Threshold': threshold, 'Precision': p, 'Recall': r, 'F1': f}

LightGBM features

In [ ]:
target = 'Is Laundering'
features = [
    'Amount Received',
    'Received Currency',
    'Payment Format',
    'hour',
    'day_of_week',
    'sender_out_count_24h',
    'receiver_in_count_24h',
    'sender_out_rate_lifetime',
    'receiver_in_rate_lifetime',
]

lgb_df = df[features + [target, 'split']].astype({
    'Received Currency': 'category',
    'Payment Format': 'category',
})
train, val, test = (lgb_df[lgb_df['split'] == s] for s in ['train', 'val', 'test'])
print(f'Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}')

LightGBM training and results

In [ ]:
lgb_path = OUT_DIR / f'lightgbm_seed{SEED}' / 'model.txt'

if lgb_path.exists():
    print('Loading saved model...')
    lgbm = lgb.Booster(model_file=lgb_path)
else:
    clf = lgb.LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=31,
        random_state=SEED,
        metric='average_precision',
    )
    clf.fit(
        train[features],
        train[target],
        eval_set=[(val[features], val[target])],
        callbacks=[lgb.early_stopping(stopping_rounds=50)],
    )
    lgbm = clf.booster_
    lgb_path.parent.mkdir(parents=True, exist_ok=True)
    lgbm.save_model(lgb_path)

lgb_val_prob = lgbm.predict(val[features])
lgb_test_prob = lgbm.predict(test[features])
lgb_result = report('LightGBM', val[target], lgb_val_prob, test[target], lgb_test_prob)

GIN model

In [ ]:
# GIN model from Multi-GNN models.py
class GINe(nn.Module):
    def __init__(self, num_features, num_gnn_layers, n_classes, n_hidden, edge_dim, final_dropout):
        super().__init__()
        self.n_hidden = n_hidden
        self.node_emb = nn.Linear(num_features, n_hidden)
        self.edge_emb = nn.Linear(edge_dim, n_hidden)
        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        for _ in range(num_gnn_layers):
            self.convs.append(GINEConv(nn.Sequential(
                nn.Linear(n_hidden, n_hidden),
                nn.ReLU(),
                nn.Linear(n_hidden, n_hidden),
            ), edge_dim=n_hidden))
            self.batch_norms.append(BatchNorm(n_hidden))
        self.mlp = nn.Sequential(
            Linear(n_hidden * 3, 50), nn.ReLU(), nn.Dropout(final_dropout),
            Linear(50, 25), nn.ReLU(), nn.Dropout(final_dropout),
            Linear(25, n_classes),
        )

    def forward(self, x, edge_index, edge_attr):
        x = self.node_emb(x)
        edge_attr = self.edge_emb(edge_attr)
        for conv, batch_norm in zip(self.convs, self.batch_norms):
            x = (x + F.relu(batch_norm(conv(x, edge_index, edge_attr)))) / 2
        x = x[edge_index.T].reshape(-1, 2 * self.n_hidden).relu()
        return self.mlp(torch.cat((x, edge_attr), 1))

GIN helpers

In [ ]:
# z_norm from Multi-GNN data_util.py
# complete_batch and predict from Multi-GNN train_util.py
def z_norm(data):
    std = data.std(0, keepdim=True)
    std[std == 0] = 1
    return (data - data.mean(0, keepdim=True)) / std

def complete_batch(batch, seed_ids, data):
    batch_ids = batch.edge_attr[:, 0]
    mask = torch.isin(batch_ids, seed_ids)
    missing = seed_ids[~torch.isin(seed_ids, batch_ids)].long()
    if len(missing) > 0:
        node_pos = torch.full((data.num_nodes,), -1)
        node_pos[batch.n_id] = torch.arange(len(batch.n_id))
        batch.edge_index = torch.cat([batch.edge_index, node_pos[data.edge_index[:, missing]]], dim=1)
        batch.edge_attr = torch.cat([batch.edge_attr, data.edge_attr[missing]], dim=0)
        batch.y = torch.cat([batch.y, data.y[missing]])
        mask = torch.cat([mask, torch.ones(len(missing), dtype=torch.bool)])
    batch.edge_attr = batch.edge_attr[:, 1:]
    return mask


@torch.no_grad()
def predict(model, loader, inds, data):
    model.eval()
    probs, labels = [], []
    for batch in loader:
        mask = complete_batch(batch, data.edge_attr[inds[batch.input_id], 0], data)
        batch, mask = batch.to(device), mask.to(device)
        out = model(batch.x, batch.edge_index, batch.edge_attr)
        probs.append(torch.softmax(out, dim=-1)[mask, 1].cpu())
        labels.append(batch.y[mask].cpu())
    return torch.cat(probs).numpy(), torch.cat(labels).numpy()

Graphs and loaders

In [ ]:
# graphs from Multi-GNN data_loading.py
# edge ids and loaders from Multi-GNN train_util.py
edge_features = ['Timestamp', 'Amount Received', 'Received Currency', 'Payment Format']
n_train = int((df['split'] == 'train').sum())
n_val = int((df['split'] == 'val').sum())
num_nodes = int(max(df['from_id'].max(), df['to_id'].max())) + 1

x = z_norm(torch.ones(num_nodes, 1))
edge_index = torch.tensor(df[['from_id', 'to_id']].to_numpy().T)
edge_attr = torch.tensor(df[edge_features].to_numpy(dtype=np.float32)).contiguous()
y = torch.tensor(df['Is Laundering'].to_numpy())

graphs = {}
for split, end in [('train', n_train), ('val', n_train + n_val), ('test', len(df))]:
    ids = torch.arange(end).view(-1, 1)
    graphs[split] = Data(
        x=x,
        edge_index=edge_index[:, :end],
        edge_attr=torch.cat([ids, z_norm(edge_attr[:end])], dim=1),
        y=y[:end],
    )
tr, va, te = graphs['train'], graphs['val'], graphs['test']
val_inds = torch.arange(n_train, n_train + n_val)
te_inds = torch.arange(n_train + n_val, len(df))

tr_loader = LinkNeighborLoader(tr, num_neighbors=[100, 100], batch_size=8192, shuffle=True)
val_loader = LinkNeighborLoader(
    va,
    num_neighbors=[100, 100],
    edge_label_index=va.edge_index[:, val_inds],
    edge_label=va.y[val_inds],
    batch_size=8192,
    shuffle=False,
)
te_loader = LinkNeighborLoader(
    te,
    num_neighbors=[100, 100],
    edge_label_index=te.edge_index[:, te_inds],
    edge_label=te.y[te_inds],
    batch_size=8192,
    shuffle=False,
)
print(f'Nodes: {num_nodes:,} | Train edges: {n_train:,} | Val edges: {n_val:,} | Test edges: {len(te_inds):,}')

GIN training

In [ ]:
# tuned GIN settings from Multi-GNN model_settings.json
# training loop from Multi-GNN training.py
lr = 0.006213266113989207
n_hidden = 66
n_gnn_layers = 2
class_weights = [1.0000182882773443, 6.275014431494497]  
final_dropout = 0.10527690625126304

torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

model = GINe(
    num_features=1,
    num_gnn_layers=n_gnn_layers,
    n_classes=2,
    n_hidden=n_hidden,
    edge_dim=len(edge_features),
    final_dropout=final_dropout,
).to(device)

gin_path = OUT_DIR / f'gin_seed{SEED}' / 'model.pt'

if gin_path.exists():
    print('Loading saved model...')
    model.load_state_dict(torch.load(gin_path, map_location=device, weights_only=True))
else:
    gin_path.parent.mkdir(parents=True, exist_ok=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, device=device))

    best_f1, best_epoch = -1.0, 0
    for epoch in range(100):
        model.train()
        losses = []
        for batch in tr_loader:
            optimizer.zero_grad()
            mask = complete_batch(batch, tr.edge_attr[batch.input_id, 0], tr)
            batch, mask = batch.to(device), mask.to(device)
            out = model(batch.x, batch.edge_index, batch.edge_attr)
            loss = loss_fn(out[mask], batch.y[mask])
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        val_prob, val_y = predict(model, val_loader, val_inds, va)
        val_f1 = f1_score(val_y, (val_prob > 0.5).astype(int), zero_division=0)
        print(f'Epoch {epoch:03d} | train loss {np.mean(losses):.4f} | val F1 {val_f1:.4f}')

        if val_f1 > best_f1:
            best_f1, best_epoch = val_f1, epoch
            torch.save(model.state_dict(), gin_path)
        if epoch - best_epoch >= 20:
            print(f'No improvement for 20 epochs, stopping at epoch {epoch}')
            break

    print(f'Best epoch: {best_epoch} (val F1 {best_f1:.4f})')
    model.load_state_dict(torch.load(gin_path, map_location=device, weights_only=True))

GIN results

In [ ]:
# fixed seed
torch.manual_seed(SEED)
gin_val_prob, gin_val_y = predict(model, val_loader, val_inds, va)
gin_test_prob, gin_test_y = predict(model, te_loader, te_inds, te)
print(f'Test edges scored: {len(gin_test_prob):,} of {len(te_inds):,}')
gin_result = report('GIN', gin_val_y, gin_val_prob, gin_test_y, gin_test_prob)

Comparison

In [ ]:
print(pd.DataFrame([lgb_result, gin_result]).set_index('Model').round(4))